In [ ]:
#@title 1. เตรียมระบบและโหลดโมเดล 4x-UltraSharp (กดรันครั้งเดียว)
!pip install -q --upgrade gradio spandrel opencv-python-headless torch torchvision pillow
!wget -q --show-progress https://huggingface.co -O 4x-UltraSharp.pth

import torch
from spandrel import ModelLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ModelLoader().load_from_file('4x-UltraSharp.pth')
model.to(device).eval()
print(f'โหลดโมเดล 4x-UltraSharp บน {device} เรียบร้อย พร้อมใช้งาน!')

In [ ]:
#@title 2. เปิดหน้าต่างคิวอัตโนมัติ (เพิ่มรูปได้ตลอดเวลา -> เข้าคิว -> โหลดลงเครื่องเอง)
import gradio as gr
import cv2
import numpy as np
import os
import shutil
import time
import threading
import queue
from collections import deque

os.makedirs('input_queue', exist_ok=True)
os.makedirs('output_done', exist_ok=True)

task_queue = queue.Queue()
pending_names = deque()
current_task = 'ว่าง (พร้อมรับรูป)'
completed_files = []
lock = threading.Lock()

def upscale_tiled(model, tensor, tile_size=512, overlap=32):
    b, c, h, w = tensor.shape
    scale = model.scale
    output = torch.zeros((b, c, h * scale, w * scale), device=tensor.device)
    weight = torch.zeros_like(output)
    for y in range(0, h, tile_size - overlap):
        for x in range(0, w, tile_size - overlap):
            y_end = min(y + tile_size, h)
            x_end = min(x + tile_size, w)
            y_start = max(0, y_end - tile_size)
            x_start = max(0, x_end - tile_size)
            tile = tensor[:, :, y_start:y_end, x_start:x_end]
            with torch.no_grad():
                out_tile = model(tile)
            output[:, :, y_start*scale:y_end*scale, x_start*scale:x_end*scale] += out_tile
            weight[:, :, y_start*scale:y_end*scale, x_start*scale:x_end*scale] += 1.0
    return output / weight

def worker():
    global current_task
    while True:
        filepath, orig_name = task_queue.get()
        with lock:
            if orig_name in pending_names:
                pending_names.remove(orig_name)
            current_task = f'กำลังขยายภาพ: {orig_name}'
        try:
            img = cv2.imread(filepath, cv2.IMREAD_COLOR)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img_tensor = torch.from_numpy(img).permute(2, 0, 1).float().div(255.0).unsqueeze(0).to(device)
                with torch.no_grad():
                    out_tensor = upscale_tiled(model, img_tensor, tile_size=512, overlap=32)
                out_img = out_tensor.squeeze(0).permute(1, 2, 0).clamp(0, 1).mul(255).byte().cpu().numpy()
                out_img = cv2.cvtColor(out_img, cv2.COLOR_RGB2BGR)
                out_name = f'4x_{os.path.splitext(orig_name)[0]}.png'
                out_path = os.path.join('output_done', out_name)
                cv2.imwrite(out_path, out_img)
                with lock:
                    completed_files.append(out_path)
        except Exception as e:
            print(f'Error processing {orig_name}: {e}')
        finally:
            with lock:
                current_task = 'ว่าง (พร้อมรับรูป)'
            task_queue.task_done()

threading.Thread(target=worker, daemon=True).start()

def add_files(files):
    if not files:
        return None, get_status(), gr.update()
    for f in files:
        src = f.name if hasattr(f, 'name') else f
        orig_name = os.path.basename(src)
        dst = os.path.join('input_queue', f'{int(time.time()*1000)}_{orig_name}')
        shutil.copy(src, dst)
        with lock:
            pending_names.append(orig_name)
        task_queue.put((dst, orig_name))
    return None, get_status(), gr.update(value=completed_files)

def get_status():
    with lock:
        q_list = list(pending_names)
        status = f'สถานะปัจจุบัน: {current_task}\n'
        status += f'รอในคิว ({len(q_list)} ภาพ): ' + (', '.join(q_list) if q_list else 'ไม่มี')
    return status

def refresh_ui():
    with lock:
        files_copy = list(completed_files)
    return get_status(), files_copy

auto_download_js = '''
() => {
    window._downloadedSet = window._downloadedSet || new Set();
    setInterval(() => {
        const links = document.querySelectorAll('#done_files a[download]');
        links.forEach(a => {
            const href = a.getAttribute('href');
            if (href && !window._downloadedSet.has(href)) {
                window._downloadedSet.add(href);
                a.click();
            }
        });
    }, 1500);
}
'''

with gr.Blocks(js=auto_download_js) as demo:
    gr.Markdown('### Auto-Queue 4x Upscaler (เลือกเพิ่มได้ตลอด -> เข้าคิว -> โหลดลงเครื่องอัตโนมัติ)')
    gr.Markdown('*แนะนำ: กดเปิดลิงก์ `https://gradio.live` ด้านล่างในแท็บใหม่ เพื่อให้เบราว์เซอร์ไม่บล็อกการดาวน์โหลดหลายไฟล์ต่อเนื่อง*')
    file_in = gr.File(label='1. กดเลือกภาพหรือลากมาวางที่นี่ได้เรื่อยๆ (เพิ่มแทรกคิวได้ตลอดเวลา)', file_count='multiple', file_types=['image'])
    status_box = gr.Textbox(label='2. สถานะคิวงาน', value=get_status(), lines=3, interactive=False)
    file_out = gr.File(label='3. ไฟล์ที่เสร็จแล้ว (ระบบจะสั่งดาวน์โหลดอัตโนมัติ หรือกดโหลดซ้ำตรงนี้ได้)', elem_id='done_files', interactive=False)

    file_in.upload(fn=add_files, inputs=[file_in], outputs=[file_in, status_box, file_out])
    timer = gr.Timer(2)
    timer.tick(fn=refresh_ui, outputs=[status_box, file_out])

demo.queue().launch(share=True, inline=True)